# Emotion Embeddings

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel


INPUT_FILE = 'cleaned_data_for_training.csv'
EMOTION_MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"
OUTPUT_FILE = 'emotion_embeddings.npy'

C:\Users\User\Desktop\FYP_Coding\fyp_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_emotion_features():
    print(f"Loading data from '{INPUT_FILE}'...")
    df = pd.read_csv(INPUT_FILE)
    texts = df['clean_text'].tolist()
    
    print(f"Loading Model: {EMOTION_MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL_NAME)
    model = AutoModel.from_pretrained(EMOTION_MODEL_NAME)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model.to(device)
    model.eval() 
    
    features_list = []
    
    print("Starting extraction...")
    
    batch_size = 32
    total = len(texts)
    
    with torch.no_grad(): 
        for i in range(0, total, batch_size):
            batch_texts = texts[i : i + batch_size]
            
            inputs = tokenizer(
                batch_texts, 
                padding=True, 
                truncation=True, 
                max_length=512, 
                return_tensors="pt"
            ).to(device)
            
            
            outputs = model(**inputs)
            
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            features_list.append(cls_embeddings)
            
            if i % 100 == 0:
                print(f"Processed {i}/{total} reviews...")

    final_features = np.vstack(features_list)
    print(f"Extraction complete. Shape: {final_features.shape}")
    
    if final_features.shape == (1600, 768):
        np.save(OUTPUT_FILE, final_features)
        print(f"✅ SUCCESS: Saved embeddings to '{OUTPUT_FILE}'")
    else:
        print(f"⚠️ WARNING: Unexpected shape {final_features.shape}.")



In [3]:
if __name__ == "__main__":
    extract_emotion_features()

Loading data from 'cleaned_data_for_training.csv'...
Loading Model: j-hartmann/emotion-english-distilroberta-base...


C:\Users\User\Desktop\FYP_Coding\fyp_venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. F

Using device: cuda
Starting extraction...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Processed 0/1600 reviews...
Processed 800/1600 reviews...
Extraction complete. Shape: (1600, 768)
✅ SUCCESS: Saved 768-dim embeddings to 'emotion_embeddings.npy'


# ABSA Embeddings

In [9]:
INPUT_FILE = 'cleaned_data_for_training.csv'
OUTPUT_FILE = 'absa_embeddings.npy'

ABSA_MODEL_NAME = "yangheng/deberta-v3-base-absa-v1.1"

def extract_absa_features():
    print(f"Loading data from '{INPUT_FILE}'...")
    df = pd.read_csv(INPUT_FILE)
    texts = df['clean_text'].tolist()
    expected_count = len(texts)
    
    print(f"Loading ABSA Model: {ABSA_MODEL_NAME}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(ABSA_MODEL_NAME)
        model = AutoModel.from_pretrained(ABSA_MODEL_NAME)
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model.to(device)
    model.eval()
    
    features_list = []
    print("Starting ABSA extraction...")
    
    batch_size = 32
    
    with torch.no_grad():
        for i in range(0, expected_count, batch_size):
            batch_texts = texts[i : i + batch_size]
            
            try:
                inputs = tokenizer(
                    batch_texts, 
                    padding=True, 
                    truncation=True, 
                    max_length=512, 
                    return_tensors="pt"
                ).to(device)

                outputs = model(**inputs)
                
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                features_list.append(batch_embeddings)
                
            except Exception as e:
                print(f"⚠️ Error at batch {i}: {e}")
                print("   -> Inserting ZERO vectors to keep alignment.")
                zero_batch = np.zeros((len(batch_texts), 768))
                features_list.append(zero_batch)

            if i % 100 == 0:
                print(f"Processed {i}/{expected_count}...")

    final_features = np.vstack(features_list)
    
    print("-" * 30)
    print(f"CSV Rows: {expected_count}")
    print(f"Embeddings: {final_features.shape[0]}")
    
    if final_features.shape == (expected_count, 768):
        np.save(OUTPUT_FILE, final_features)
        print(f"✅ SUCCESS: Saved ABSA embeddings to '{OUTPUT_FILE}'")
    else:
        print(f"❌ ERROR: Row mismatch.")

if __name__ == "__main__":
    extract_absa_features()

Loading data from 'cleaned_data_for_training.csv'...
Loading ABSA Model: yangheng/deberta-v3-base-absa-v1.1...


C:\Users\User\Desktop\FYP_Coding\fyp_venv\lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Using device: cuda
Starting ABSA extraction... (This takes time)
Processed 0/1600...
Processed 800/1600...
------------------------------
CSV Rows: 1600
Embeddings: 1600
✅ SUCCESS: Saved ABSA embeddings to 'absa_embeddings.npy'


# Emotion Model Evaluation

In [2]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Loading dataset...")
df = pd.read_csv('cleaned_data_for_training.csv')

df = df.dropna(subset=['clean_text', 'polarity'])
texts = df['clean_text'].tolist()
true_sentiments = df['polarity'].str.lower().tolist() 

print("Loading Emotion Model...")
emotion_classifier = pipeline(
    "text-classification", 
    model="j-hartmann/emotion-english-distilroberta-base", 
    top_k=1, 
    device=0 
)

def map_emotion_to_sentiment(emotion):
    if emotion in ['anger', 'disgust', 'fear', 'sadness']:
        return 'negative'
    elif emotion == 'joy':
        return 'positive'
    else:
        return 'ignore' 

print(f"Extracting emotions for {len(texts)} reviews...")
mapped_predictions = []
valid_true_sentiments = []

batch_size = 32

for i in range(0, len(texts), batch_size):
    if i % (batch_size * 5) == 0:
        print(f"Processing row {i}/{len(texts)}...")
        
    batch_texts = texts[i:i+batch_size]
    

    batch_results = emotion_classifier(batch_texts, truncation=True, max_length=512)
    
    for j, result in enumerate(batch_results):
        predicted_emotion = result[0]['label']
        mapped_sentiment = map_emotion_to_sentiment(predicted_emotion)
        
        if mapped_sentiment != 'ignore':
            mapped_predictions.append(mapped_sentiment)
            valid_true_sentiments.append(true_sentiments[i+j])

print("\n" + "="*60)
print("EMOTION MODEL VALENCE-MAPPING EVALUATION RESULTS")
print("="*60)
accuracy = accuracy_score(valid_true_sentiments, mapped_predictions)
print(f"Total reviews evaluated: {len(valid_true_sentiments)}")
print(f"Mapped Accuracy: {accuracy:.4f}\n")

print("--- Classification Report ---")
print(classification_report(valid_true_sentiments, mapped_predictions))

print("--- Confusion Matrix ---")
print(confusion_matrix(valid_true_sentiments, mapped_predictions))

Loading dataset...
Loading Emotion Model...


Device set to use cuda:0


Extracting emotions for 1600 reviews...
Processing row 0/1600...
Processing row 160/1600...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processing row 320/1600...
Processing row 480/1600...
Processing row 640/1600...
Processing row 800/1600...
Processing row 960/1600...
Processing row 1120/1600...
Processing row 1280/1600...
Processing row 1440/1600...

EMOTION MODEL VALENCE-MAPPING EVALUATION RESULTS
Total reviews evaluated: 1070
Mapped Accuracy: 0.9617

--- Classification Report ---
              precision    recall  f1-score   support

    negative       0.96      0.96      0.96       506
    positive       0.96      0.96      0.96       564

    accuracy                           0.96      1070
   macro avg       0.96      0.96      0.96      1070
weighted avg       0.96      0.96      0.96      1070

--- Confusion Matrix ---
[[485  21]
 [ 20 544]]


# ABSA model analysis

In [4]:
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
import pandas as pd
from collections import Counter
from pyabsa import AspectTermExtraction as ATEPC

INPUT_FILE = 'cleaned_data_for_training.csv'
ABSA_MODEL_NAME = "english"
SAMPLE_SIZE = None

def analyze_corpus(texts, extractor, label_name):
    print(f"\nAnalyzing {len(texts)} {label_name} reviews...")
    
    total_aspects = 0
    reviews_with_zero_aspects = 0
    aspect_counter = Counter()
    sentiment_counter = Counter()
    
    for i, text in enumerate(texts):
        if i % 100 == 0 and i > 0:
            print(f"  Processed {i}/{len(texts)}...")
            
        result = extractor.predict(text, print_result=False)
        
        if isinstance(result, list):
            result = result[0]
            
        aspects = result.get('aspect', [])
        sentiments = result.get('sentiment', [])
        
        if not aspects:
            reviews_with_zero_aspects += 1
        else:
            total_aspects += len(aspects)
            aspect_counter.update([str(a).lower() for a in aspects])
            sentiment_counter.update(sentiments)
            

    avg_aspects = total_aspects / len(texts)
    
    print("\n" + "="*40)
    print(f" {label_name.upper()} REVIEW STATISTICS")
    print("="*40)
    print(f"Total Reviews Analyzed:   {len(texts)}")
    print(f"Total Aspects Found:      {total_aspects}")
    print(f"Average Aspects/Review:   {avg_aspects:.2f}")
    print(f"Reviews w/ NO Aspects:    {reviews_with_zero_aspects} ({(reviews_with_zero_aspects/len(texts))*100:.1f}%)")
    
    print("\nTop 10 Most Mentioned Aspects:")
    for aspect, count in aspect_counter.most_common(10):
        print(f"  -> '{aspect}': {count} times")
        
    print("\nSentiment Distribution:")
    for sent, count in sentiment_counter.items():
        percentage = (count / total_aspects) * 100 if total_aspects > 0 else 0
        print(f"  -> {sent.capitalize()}: {count} ({percentage:.1f}%)")

def run_full_analysis():
    print("1. Loading Data...")
    df = pd.read_csv(INPUT_FILE)
    
    df_genuine = df[df['deceptive_flag'] == 0]['clean_text'].tolist()
    df_fraud = df[df['deceptive_flag'] == 1]['clean_text'].tolist()
    
    if SAMPLE_SIZE is not None:
        print(f"Limiting to {SAMPLE_SIZE} reviews per class for speed...")
        df_genuine = df_genuine[:SAMPLE_SIZE]
        df_fraud = df_fraud[:SAMPLE_SIZE]
        
    print(f"2. Loading ABSA Model ({ABSA_MODEL_NAME})...")
    extractor = ATEPC.AspectExtractor(ABSA_MODEL_NAME, auto_device=True)
    
    analyze_corpus(df_genuine, extractor, "Genuine")
    analyze_corpus(df_fraud, extractor, "Fraudulent")

if __name__ == "__main__":
    run_full_analysis()

1. Loading Data...
2. Loading ABSA Model (english)...
[2026-03-01 17:18:19] (2.4.3) ********** Available ATEPC model checkpoints for Version:2.4.3 (this version) **********
[2026-03-01 17:18:19] (2.4.3) ********** Available ATEPC model checkpoints for Version:2.4.3 (this version) **********
[2026-03-01 17:18:19] (2.4.3) Downloading checkpoint:english 
[2026-03-01 17:18:19] (2.4.3) Notice: The pretrained model are used for testing, it is recommended to train the model on your own custom datasets
[2026-03-01 17:18:19] (2.4.3) Checkpoint already downloaded, skip
[2026-03-01 17:18:19] (2.4.3) Load aspect extractor from checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.89_atef1_75.43
[2026-03-01 17:18:19] (2.4.3) config: checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.89_atef1_75.43\fast_lcf_atepc.config
[2026-03-01 17:18:19] (2.4.3) state_dict: checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82

C:\Users\User\Desktop\FYP_Coding\fyp_venv\lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(



Analyzing 800 Genuine reviews...
[2026-03-01 17:18:26] (2.4.3) The results of aspect term extraction have been saved in C:\Users\User\Desktop\FYP_Coding\FYP\Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-01 17:18:28] (2.4.3) The results of aspect term extraction have been saved in C:\Users\User\Desktop\FYP_Coding\FYP\Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-01 17:18:30] (2.4.3) The results of aspect term extraction have been saved in C:\Users\User\Desktop\FYP_Coding\FYP\Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-01 17:18:32] (2.4.3) The results of aspect term extraction have been saved in C:\Users\User\Desktop\FYP_Coding\FYP\Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
[2026-03-01 17:18:34] (2.4.3) The results of aspect term extraction have been saved in C:\Users\User\Desktop\FYP_Coding\FYP\Aspect Term Extraction and Polarit